In [1]:
# 导入需要的包
# Import the required packages.
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import random
import csv
from tqdm import tqdm
import zipfile
import pandas as pd
import pandas as pd
from PIL import Image
import sys
sys.path.insert(0, "/bohr/data-x5e5/v2")
from dataset import CustomDataset

In [ ]:
# data loading
train_dir = '/bohr/data-x5e5/v2/train/'  #数据地址 #Address of dataset
train_file = '/bohr/data-x5e5/v2/train.csv' #训练集标注地址 #Address of training data annotations
train_ds = pd.read_csv(train_file, encoding = "GB2312")
cloth_ds = train_ds[train_ds["1"] == 1]
make_ds = train_ds[train_ds["1"] == 0]

In [ ]:
len(cloth_ds), len(make_ds)

In [ ]:
import random
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

X_col = train_ds.columns[0]
y_col  = train_ds.columns[2]

def show_random_images(ds, title, n=100):
    samples = ds[X_col].sample(n=min(n, len(ds)), random_state=random.randint(0, 9999)).tolist()
    cols = 10
    rows = 10
    fig, axes = plt.subplots(rows, cols, figsize=(20, 20))
    fig.suptitle(title, fontsize=16, y=1.01)
    
    for i, ax in enumerate(axes.flatten()):
        if i < len(samples):
            try:
                img = mpimg.imread(os.path.join(train_dir, samples[i] + ".jpg"))
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, 'Error', ha='center', va='center')
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

show_random_images(cloth_ds, "Cloth Images (label=1)", n=100)

In [ ]:
show_random_images(make_ds, "Make Images (label=0)", n=100)

In [ ]:
import random
import cv2
import numpy as np
import os
import pandas as pd

def four_img_stack(img1, img2, img3, img4):
    h, w = 224, 224
    half = h // 2
    img1 = cv2.resize(img1, (half, half))
    img2 = cv2.resize(img2, (half, half))
    img3 = cv2.resize(img3, (half, half))
    img4 = cv2.resize(img4, (half, half))
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[0:half, 0:half] = img1
    img[0:half, half:w] = img2
    img[half:h, 0:half] = img3
    img[half:h, half:w] = img4
    return img

def two_img_stack(img1, img2):
    h, w = 224, 224
    half = h // 2
    img1 = cv2.resize(img1, (w, half))
    img2 = cv2.resize(img2, (w, half))
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[0:half, 0:w] = img1
    img[half:h, 0:w] = img2
    return img

make_grid_rows = []
failed = []
out_dir = "/root/grid_images"
os.makedirs(out_dir, exist_ok=True)

for idx, row in make_ds.iterrows():
    img_name = row[X_col]
    out_filename = img_name + "_grid.jpg"
    out_path = os.path.join(out_dir, out_filename)

    # --- save original image with label = 0 ---
    orig_src = os.path.join(train_dir, img_name + ".jpg")
    orig_dst = os.path.join(out_dir,   img_name + ".jpg")
    orig_img = cv2.imread(orig_src)
    if orig_img is not None:
        cv2.imwrite(orig_dst, orig_img)
        orig_row = row.copy()
        orig_row[X_col] = img_name + ".jpg"
        orig_row[y_col] = 0                     # label = 0 for original
        make_grid_rows.append(orig_row)
    else:
        print(f"[WARN] Could not read original: {orig_src}")

    # --- build grid image with label = 1 ---
    if random.random() > 0.5:
        val1 = random.randint(0, len(make_ds) - 1)
        val2 = random.randint(0, len(make_ds) - 1)
        val3 = random.randint(0, len(make_ds) - 1)
        val4 = random.randint(0, len(make_ds) - 1)
        img1 = cv2.imread(os.path.join(train_dir, make_ds.iloc[val1][X_col] + ".jpg"))
        img2 = cv2.imread(os.path.join(train_dir, make_ds.iloc[val2][X_col] + ".jpg"))
        img3 = cv2.imread(os.path.join(train_dir, make_ds.iloc[val3][X_col] + ".jpg"))
        img4 = cv2.imread(os.path.join(train_dir, make_ds.iloc[val4][X_col] + ".jpg"))
        new_img = four_img_stack(img1, img2, img3, img4)
    else:
        val1 = random.randint(0, len(make_ds) - 1)
        val2 = random.randint(0, len(make_ds) - 1)
        img1 = cv2.imread(os.path.join(train_dir, make_ds.iloc[val1][X_col] + ".jpg"))
        img2 = cv2.imread(os.path.join(train_dir, make_ds.iloc[val2][X_col] + ".jpg"))
        new_img = two_img_stack(img1, img2)

    success = cv2.imwrite(out_path, new_img)
    if success:
        grid_row = row.copy()
        grid_row[X_col] = out_filename
        grid_row[y_col] = 1                     # label = 1 for grid image
        make_grid_rows.append(grid_row)
    else:
        print(f"[FAIL] {out_filename}")
        failed.append(out_filename)

make_grid_ds = pd.DataFrame(make_grid_rows, columns=make_ds.columns)
make_grid_ds.to_csv(os.path.join(out_dir, "make_grid_ds.csv"), index=False)
print(f"\nDone. Total rows: {len(make_grid_ds)}, Failed: {len(failed)}")
print(f"Label counts:\n{make_grid_ds[y_col].value_counts()}")
print(f"All files saved to: {out_dir}")

In [ ]:
make_grid_ds

In [ ]:
import shutil

cloth_rows = []

for idx, row in cloth_ds.iterrows():
    src_path = os.path.join(train_dir, row[X_col] + ".jpg")
    out_path = os.path.join(out_dir, row[X_col] + ".jpg")
    
    shutil.copy(src_path, out_path)
    
    new_row = row.copy()
    new_row[X_col] = row[X_col] + ".jpg"
    new_row[y_col] = 1
    cloth_rows.append(new_row)        
cloth_ds_new = pd.DataFrame(cloth_rows, columns=cloth_ds.columns)
print(f"Copied {len(cloth_ds_new)} cloth images to {out_dir}")
cloth_ds_new.head()

In [ ]:
grid_ds= pd.concat([make_grid_ds, cloth_ds_new], axis = 0)
final_ds = grid_ds.reset_index(drop = True)

In [ ]:
fig, axes = plt.subplots(10,5, figsize = (20, 20))
axes = axes.flatten()
for idx, row in final_ds[:50].iterrows():
    img = Image.open(os.path.join(out_dir, row[X_col]))
    axes[idx].imshow(img)
    axes[idx].set_title(row["1"])
plt.show()

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
from PIL import Image
import pandas as pd
import numpy as np
import cv2
import os
model = YOLO("yolov8n.pt")
single_cloth_rows = []
PERSON_CLASS = 0
for idx, row in cloth_ds.iterrows():
    img_name = row[X_col]
    img_path = os.path.join(train_dir, img_name + ".jpg")
    if not os.path.exists(img_path):
        continue
    try:
        img = Image.open(img_path).convert("RGB")
        img_np = np.array(img)
    except Exception:
        continue
    h, w = img_np.shape[:2]
    result = model.predict(
        source=img_np,
        conf=0.25,
        verbose=False
    )[0]
    boxes = result.boxes
    if len(boxes) == 0:
        continue
    crops = []
    for box in boxes:
        cls_id = int(box.cls[0])
        if cls_id != PERSON_CLASS:
            continue
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        x1 = max(0, int(x1))
        y1 = max(0, int(y1))
        x2 = min(w, int(x2))
        y2 = min(h, int(y2))
        if x2 <= x1 or y2 <= y1:
            continue
        area = (x2 - x1) * (y2 - y1)
        if area < 5000:
            continue
        crop = img_np[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)

        if gray.std() < 10:
            continue
        crops.append(crop)
    if len(crops) == 0:
        continue
    for i, crop in enumerate(crops):
        new_name = f"{img_name}_{i}.jpg"
        out_path = os.path.join(out_dir, new_name)

        success = cv2.imwrite(
            out_path,
            cv2.cvtColor(crop, cv2.COLOR_RGB2BGR)
        )
        if not success:
            continue
        new_row = row.copy()
        new_row[X_col] = new_name.replace(".jpg", "")
        single_cloth_rows.append(new_row)
single_cloth_df = pd.DataFrame(single_cloth_rows)
print(f"Generated {len(single_cloth_df)} cropped samples")

In [ ]:
single_cloth_df = single_cloth_df.reset_index(drop = True)
single_cloth_df["1"] = 0
single_cloth_df[X_col]= single_cloth_df[X_col] + ".jpg"


In [ ]:
final_ds = pd.concat([final_ds, single_cloth_df], axis= 0)

In [ ]:
final_ds = final_ds.reset_index(drop = True)
final_ds = final_ds.sample(frac=1).reset_index(drop=True)

In [ ]:
fig, axes = plt.subplots(10,5, figsize = (20, 20))
axes = axes.flatten()
for idx, row in final_ds[:50].iterrows():
    img = Image.open(os.path.join(out_dir, row[X_col]))
    axes[idx].imshow(img)
    axes[idx].set_title(row["1"])
plt.show()

In [ ]:
from torchvision.models import resnet34
class DF(Dataset):
    def __init__(self, path, ds,trans, ):
        self.path = path
        self.trans = trans
        self.ds = ds
        self.imgs= self.ds[X_col]
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.path, self.imgs[idx])).convert("RGB")
        img = self.trans(img)
        label = self.ds[y_col][idx]
        return img, torch.tensor(label, dtype = torch.long)
from torchvision import transforms as T
import torchvision.transforms.functional as TF

train_trans = T.Compose([
    T.Resize((224, 224)), 
    T.RandomResizedCrop(
        224,
        scale=(0.6, 1.0),    
        ratio=(0.75, 1.33)
    ),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.3),  
    T.RandomApply([
        T.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.5,
            hue=0.2
        )
    ], p=0.8),

    T.RandomRotation(degrees=45),
    T.RandomPerspective(distortion_scale=0.5, p=0.5),
    T.RandomApply([
        T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
    ], p=0.4),
    T.RandomGrayscale(p=0.1),

    T.ToTensor(),
    T.RandomErasing(
        p=0.5,
        scale=(0.02, 0.2),
        ratio=(0.3, 3.3),
        value="random"
    ),

    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
train_df= DF(out_dir, final_ds, train_trans)
train_dl = DataLoader(train_df, batch_size = 16, shuffle = True)
for batch in train_dl:
    x, y= batch
    print(x.shape)
    print(y.shape)
    break
class MODEL(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = resnet34(pretrained = True)
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)
    def forward(self, x):
        return self.model(x)
device = ("cuda" if torch.cuda.is_available() else "cpu")
model = MODEL().to(device)
print(len(final_ds[final_ds["1"] == 1]))
print(len(final_ds[final_ds["1"] == 0]))

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr = 1e-5, weight_decay = 2e-5)
loss_fn = nn.CrossEntropyLoss()
for e in range(50):
    t_loss= 0
    model.train()
    for batch in train_dl:
        x, y= batch
        x, y= x.to(device), y.to(device)
        out = model(x)
        loss =loss_fn(out, y)
        t_loss += loss.item()
        opt.zero_grad()
        loss.backward()
        opt.step()
    print("E: ",e, "train_loss: ", t_loss/len(train_dl))

In [ ]:
def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc = 'Test'):
            x = batch.to(device)
            output = model(x)
            pred = torch.argmax(output, dim=1)
            preds.extend(pred.cpu().numpy())
    return preds

In [ ]:
# Save to CSV
def save_submission_csv(preds, save_name):
    df = pd.DataFrame(preds)
    df.to_csv(save_name, index=False, header=False)

In [ ]:
## 获取用于AB榜评测的验证集与测试集
## 【仅在notebook提交至比赛后可正常获取】
## 【代码调试阶段报错是正常现象，因为以下数据不对选手公开，无法在这个阶段被读取】

if os.environ.get('ANSWER_PATH'):
    PATH = os.environ.get("ANSWER_PATH") + "/" 
else:
    print("Baseline运行时，因为无法读取测试集，所以后续会有报错，属于正常现象")  

In [ ]:
# 测试阶段
val_dir = PATH + 'val/'
val_file = PATH + 'val.csv'
test_dir = PATH + 'test/'
test_file = PATH + 'test.csv'
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
# Val (val: public score, test: private score)
val_dataset = CustomDataset(val_dir, val_file, mode="val", transform=transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Test
test_dataset = CustomDataset(test_dir, test_file, mode="test", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

val_preds = predict(model, val_loader, device)
test_preds = predict(model, test_loader, device)
# Submission Process
save_submission_csv(val_preds, 'submissionA.csv')
save_submission_csv(test_preds, 'submissionB.csv')
with zipfile.ZipFile('submission.zip', 'w') as zipf:
    zipf.write('submissionA.csv')
    zipf.write('submissionB.csv')
os.remove('submissionA.csv')
os.remove('submissionB.csv')